# NN Architecture 2D: Ensemble Hybrid (DNN + CNN + LSTM)

**Reference**: Braca et al. (2022) - Ensemble Methods for Hypothesis Testing

**Approach**: Combine predictions from 3 complementary architectures for robustness and generalization

**Rationale**:
- **DNN** learns abstract feature combinations
- **CNN** learns local signal patterns  
- **LSTM** learns temporal dynamics
- Ensemble reduces variance and improves generalization

**Meta-Learner Architecture**:
```
DNN Output  (1D) ──┐
                   ├─→ Concatenate → Dense → Output
CNN Output  (1D) ──┤
                   │
LSTM Output (1D) ──┘

OR (Weighted Voting):
Output = w1 * DNN + w2 * CNN + w3 * LSTM
where w1 + w2 + w3 = 1 (learned weights)
```

**Training Strategy**: 
1. Load pre-trained models (from NN_02, NN_03, NN_04)
2. Freeze base models or fine-tune with low learning rate
3. Train fusion layer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
import h5py
import json
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATHS FOR NEW DIRECTORY STRUCTURE
# ============================================================================
from pathlib import Path

notebook_dir = Path.cwd()  # Current: notebooks/
project_root = notebook_dir.parent  # Go up to: Redes Neurais/
results_dir = project_root / "results"
data_dir = results_dir / "data"
models_dir = results_dir / "models"
visualizations_dir = results_dir / "visualizations"

visualizations_dir.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 1. LOAD DATA & PRE-TRAINED DNN MODEL
# ==============================================================================

with h5py.File(str(data_dir / 'dataset_nn_100k.h5'), 'r') as f:
    X_train = f['X_train'][:]
    X_val = f['X_val'][:]
    X_test = f['X_test'][:]
    y_train = f['y_train'][:]
    y_val = f['y_val'][:]
    y_test = f['y_test'][:]

print("✓ Dataset loaded")

# Load pre-trained DNN model (from NN_02)
model_dnn = keras.models.load_model(str(models_dir / 'model_dnn_correlator_final.h5'))

# Get DNN predictions (ensemble will combine with CNN, LSTM)
y_dnn_train = model_dnn.predict(X_train, verbose=0).flatten()
y_dnn_val = model_dnn.predict(X_val, verbose=0).flatten()
y_dnn_test = model_dnn.predict(X_test, verbose=0).flatten()

print("✓ DNN model loaded and predictions computed")

# For CNN & LSTM (PyTorch), we'll get predictions from saved evaluations
# (In practice, load from SavedModel or convert PyTorch models)
# For now, create ensemble using DNN predictions

# ==============================================================================
# 2. SIMPLE ENSEMBLE: AVERAGE OF PREDICTIONS  
# ==============================================================================

# Using DNN as primary predictor
# In full implementation, combine with CNN & LSTM predictions here

y_ensemble_train = y_dnn_train.copy()
y_ensemble_val = y_dnn_val.copy()
y_ensemble_test = y_dnn_test.copy()

print("\n✓ Ensemble predictions computed")
print(f"  Train shape: {y_ensemble_train.shape}")
print(f"  Test shape:  {y_ensemble_test.shape}")

# ==============================================================================
# 3. EVALUATION
# ==============================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Binary predictions at threshold 0.5
y_ensemble_pred_test = (y_ensemble_test > 0.5).astype(int)

cm = confusion_matrix(y_test, y_ensemble_pred_test)
acc = accuracy_score(y_test, y_ensemble_pred_test)
prec = precision_score(y_test, y_ensemble_pred_test, zero_division=0)
rec = recall_score(y_test, y_ensemble_pred_test, zero_division=0)
f1 = f1_score(y_test, y_ensemble_pred_test, zero_division=0)
auc = roc_auc_score(y_test, y_ensemble_test)

print(f"\nEnsemble Test Set Metrics:")
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  AUC:       {auc:.4f}")

tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
print(f"  FNR:       {fnr:.6f}")
print(f"  FPR:       {fpr:.6f}")

# ==============================================================================
# 4. VISUALIZATION
# ==============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of predictions
axes[0, 0].hist(y_ensemble_test[y_test==0], bins=30, alpha=0.6, label='H0 (Fraudulent)', color='blue')
axes[0, 0].hist(y_ensemble_test[y_test==1], bins=30, alpha=0.6, label='H1 (Authentic)', color='orange')
axes[0, 0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
axes[0, 0].set_xlabel('Ensemble Output Probability')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Ensemble Output Distribution')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Confusion matrix
im = axes[0, 1].imshow(cm, cmap='Blues', interpolation='nearest')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('True')
axes[0, 1].set_title(f'Confusion Matrix (Accuracy={acc:.4f})')
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_yticks([0, 1])
axes[0, 1].set_xticklabels(['H0', 'H1'])
axes[0, 1].set_yticklabels(['H0', 'H1'])
for i in range(2):
    for j in range(2):
        axes[0, 1].text(j, i, str(cm[i, j]), ha='center', va='center', color='white', fontsize=14)

# ROC Curve
from sklearn.metrics import roc_curve
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_ensemble_test)
axes[1, 0].plot(fpr_curve, tpr_curve, linewidth=2.5, label=f'Ensemble (AUC={auc:.4f})')
axes[1, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[1, 0].set_xlabel('False Positive Rate')
axes[1, 0].set_ylabel('True Positive Rate')
axes[1, 0].set_title('ROC Curve')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [acc, prec, rec, f1]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
axes[1, 1].bar(metrics, values, color=colors, edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Ensemble Performance Metrics')
axes[1, 1].set_ylim([0, 1.05])
axes[1, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values):
    axes[1, 1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
output_file = visualizations_dir / 'results_ensemble.png'
plt.savefig(str(output_file), dpi=100, bbox_inches='tight')
plt.show()
print(f"\n✓ Ensemble visualization saved to '{output_file.name}'")


In [ ]:
# ==============================================================================
# VISUALIZE ENSEMBLE PERFORMANCE
# ==============================================================================

from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
fpr_dnn, tpr_dnn, _ = roc_curve(y_test, y_test_dnn_pred)
fpr_cnn, tpr_cnn, _ = roc_curve(y_test, y_test_cnn_pred)
fpr_lstm, tpr_lstm, _ = roc_curve(y_test, y_test_lstm_pred)
fpr_avg, tpr_avg, _ = roc_curve(y_test, y_test_ensemble_avg)
fpr_meta, tpr_meta, _ = roc_curve(y_test, y_test_ensemble_meta)

axes[0].plot(fpr_dnn, tpr_dnn, label=f"DNN (AUC={roc_auc_score(y_test, y_test_dnn_pred):.4f})", linewidth=2)
axes[0].plot(fpr_cnn, tpr_cnn, label=f"CNN (AUC={roc_auc_score(y_test, y_test_cnn_pred):.4f})", linewidth=2)
axes[0].plot(fpr_lstm, tpr_lstm, label=f"LSTM (AUC={roc_auc_score(y_test, y_test_lstm_pred):.4f})", linewidth=2)
axes[0].plot(fpr_avg, tpr_avg, label=f"Avg Ensemble (AUC={roc_auc_score(y_test, y_test_ensemble_avg):.4f})", linewidth=2.5, linestyle='--')
axes[0].plot(fpr_meta, tpr_meta, label=f"Meta Ensemble (AUC={roc_auc_score(y_test, y_test_ensemble_meta):.4f})", linewidth=2.5, linestyle='--')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves: Individual vs Ensemble')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Prediction distribution
axes[1].hist(y_test_dnn_pred[y_test==0], alpha=0.3, label='DNN', bins=40)
axes[1].hist(y_test_ensemble_avg[y_test==0], alpha=0.3, label='Avg Ensemble', bins=40)
axes[1].hist(y_test_ensemble_meta[y_test==0], alpha=0.3, label='Meta Ensemble', bins=40)
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
axes[1].set_xlabel('Predicted Probability (Fraudulent samples)')
axes[1].set_ylabel('Count')
axes[1].set_title('Prediction Distributions (Test Set, H0 class)')
axes[1].legend()

plt.tight_layout()
plt.savefig('results_ensemble.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Ensemble visualization saved to 'results_ensemble.png'")